In [ ]:
# %% Minimal setup from class

import os, json, textwrap, re, time
import requests

API_KEY  = os.getenv("OPENAI_API_KEY", "cse476")
API_BASE = os.getenv("API_BASE", "http://10.4.58.53:41701/v1")  
MODEL    = os.getenv("MODEL_NAME", "bens_model")              

SYSTEM_PROMPT = "You are a helpful assistant. Reply with only the final answer—no explanation."
TEMPERATURE   = 0.45 #Must be a float

def call_model_chat_completions(prompt: str,
                                system: str = SYSTEM_PROMPT,
                                model: str = MODEL,
                                temperature: float = TEMPERATURE,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": 450,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}

def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)

def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        #1) Get model prediction
        if t.get("input"):
            r = call_model_chat_completions(
                t["input"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )
            got = reasoning_via_planning(t["input"])
            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["input"],
                prediction=got,
                expected_answer=t["output"],
                model=judge_model,
            )
        else:
            r = call_model_chat_completions(
                t["prompt"],
                system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
                model=model,
                temperature=TEMPERATURE,
            )        #got = (r.get("text") or "").strip()
            got = reasoning_via_planning(t["prompt"])

            # 2) LLM-as-a-judge: strict True/False
            is_correct = self_evaluate(
                question=t["prompt"],
                prediction=got,
                expected_answer=t["expected"],
                model=judge_model,
            )
        



        row = {
            "id": t.get("id", "<unnamed>"),
            "output": t["output"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: output={row['output']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows


In [ ]:
def reasoning_via_planning(question: str,) -> dict:
    reasoning_str = "You will decompose the problem down into a step by step plan to solve the question provided. Carefully consider each step before moving on to the next. Once you have a plan, execute each step in order to arrive at the final answer. Be sure to double-check your work at each step to ensure accuracy. Keep in mind you have limited word count to use, so be efficient and limit word output.\n\n Question: "

    r = call_model_chat_completions(
            reasoning_str + question,
            system=SYSTEM_PROMPT,
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [10]:
def tree_of_thought(question: str, n_paths: int):
    tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a reasonable path that is different from every othe path created but still leads to the answer. Keep in mind you have limited word count to use, so be efficient and limit word output. Expected output should be in the form of: path1:\{\}, \npath2:\{\},etc.\n\n Question: "
    r = call_model_chat_completions(
            tot_str.format(n_paths=n_paths) + question,
            system=SYSTEM_PROMPT,
            model=MODEL,
            temperature=TEMPERATURE,
        )
    #Divide into n_paths
    raw = r.get("text") or ""
    thoughts = []
    for n in range(1, n_paths + 1):
        path = raw.find(f"path{n}:")
        end = raw.find(f"path{n+1}:")
        if end == -1:
            end = len(raw)
        thoughts.append(raw[path:end].strip())
    return thoughts

<>:2: SyntaxWarning: invalid escape sequence '\{'
<>:2: SyntaxWarning: invalid escape sequence '\{'
C:\Users\isami\AppData\Local\Temp\ipykernel_20192\3748373568.py:2: SyntaxWarning: invalid escape sequence '\{'
  tot_str = "You will decompose the problem down into {n_paths} distinct possible solution paths to solve the question provided. Depending on the problem, write a reasonable path that is different from every othe path created but still leads to the answer. Keep in mind you have limited word count to use, so be efficient and limit word output. Expected output should be in the form of: path1:\{\}, \npath2:\{\},etc.\n\n Question: "


In [ ]:
def double_check(input: str):
    check_str = "You are a careful solver. Given the following solution, double-check each step for accuracy and correctness. If you find any mistakes, correct them and provide the accurate final answer. If everything is correct, simply confirm the final answer. Reply ONLY with the final answer—no explanation if possible.\n\n Solution to double-check: "
    r = call_model_chat_completions(
            check_str + input,
            system=SYSTEM_PROMPT,
            model=MODEL,
            temperature=TEMPERATURE,
        )
    got = (r.get("text") or "").strip()
    return got

In [11]:
import json
import random

with open("cse476_final_project_dev_data.json", "r") as tests:
    DEV_DATA = json.load(tests)

#Get test batches by domain/random
def filter_domain(domain: str):
    filtered = []
    for test in DEV_DATA:
        if test.get("domain") == domain:
            filtered.append(test)
    return filtered

def get_batch(num: int, domain: str = None, is_random: bool = False):
    random.seed(315)
    if domain:
        data = filter_domain(domain)
    else:
        data = DEV_DATA

    if is_random:
        return random.sample(data, num)
    else:
        return data[:num]
    

In [ ]:
# Tree of thought (X of thought)
# Reasoning via planning
# 'Wait' am i correct? Double check
# Critic 
# Send to output

# Future:
# Implement RAG or memory of some kind to grab from text data
# implement In-context learning with examples via RAG
# Call to Wikipedia API/ disctionary API/ Calc?

def agent_loop(input_question: str):
    #First split into thoughts
    thoughts = tree_of_thought(input_question, n_paths=3)
    #Reason through each thought path
    for t in thoughts:
        t = reasoning_via_planning(t)
    #Implement wait am i correct / double check
    
    pass

In [ ]:

# Example:
#test = reasoning_via_planning("A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")
#test2 = call_model_chat_completions("Make a plan to solve: A farmer has 17 sheep and all but 9 are lost. How many sheep are left on the farm?")
#print(test)
#print(test2)

tests = get_batch(1, domain="math", is_random=False)
self_evaluate_tests(tests, model=MODEL, sleep_sec=0.5, verbose=True)

❌ <unnamed>: output='112', got='Let the area of triangle $APB$ be $A_1$, the area of triangle $BPC$ be $A_2$, the area of triangle $CPD$ be $A_3$, and the area of triangle $APD$ be $A_4$. The total area of quadrilateral $ABCD$ is $A = A_1 + A_2 + A_3 + A_4$.\n\nGiven:\n- $AB = CD = 10$\n- $BC = 14$\n- $AD = 2\\sqrt{65}$\n- $\\text{Area of } \\triangle APB + \\text{Area of } \\triangle CPD = \\text{Area of } \\triangle BPC + \\text{Area of } \\triangle APD$\n\nThis implies:\n$$\nA_1 + A_3 = A_2 + A_4\n$$\n\nSo, the total area is:\n$$\nA = (A_1 + A_3) + (A_2 + A_4) = 2(A_1 + A_3)\n$$\n\nLet’s denote the area of triangle $APB$ as $A_1$, and the area of triangle $CPD$ as $A_3$. Then:\n$$\nA = 2(A_1 + A_3)\n$$\n\nLet’s find the area of quadrilateral $ABCD$ using the given side lengths and the fact that the diagonals divide the quadrilateral into four triangles with equal area sums.\n\nUse the formula for the area of a quadrilateral with given sides and diagonals intersecting at a point, but

[{'id': '<unnamed>',
  'output': '112',
  'got': 'Let the area of triangle $APB$ be $A_1$, the area of triangle $BPC$ be $A_2$, the area of triangle $CPD$ be $A_3$, and the area of triangle $APD$ be $A_4$. The total area of quadrilateral $ABCD$ is $A = A_1 + A_2 + A_3 + A_4$.\n\nGiven:\n- $AB = CD = 10$\n- $BC = 14$\n- $AD = 2\\sqrt{65}$\n- $\\text{Area of } \\triangle APB + \\text{Area of } \\triangle CPD = \\text{Area of } \\triangle BPC + \\text{Area of } \\triangle APD$\n\nThis implies:\n$$\nA_1 + A_3 = A_2 + A_4\n$$\n\nSo, the total area is:\n$$\nA = (A_1 + A_3) + (A_2 + A_4) = 2(A_1 + A_3)\n$$\n\nLet’s denote the area of triangle $APB$ as $A_1$, and the area of triangle $CPD$ as $A_3$. Then:\n$$\nA = 2(A_1 + A_3)\n$$\n\nLet’s find the area of quadrilateral $ABCD$ using the given side lengths and the fact that the diagonals divide the quadrilateral into four triangles with equal area sums.\n\nUse the formula for the area of a quadrilateral with given sides and diagonals intersecti